# NAM A2 → A1 converter

Converts [Neural Amp Modeler](https://github.com/sdatkinson/neural-amp-modeler) **A2** captures into **A1-standard** `.nam` files for gear and software that only read A1, such as the 1st-gen Hotone Ampero.

**How to use**
1. Choose **Runtime → Change runtime type → T4 GPU**. The CPU works too, but it is much slower.
2. Choose **Runtime → Run all**.
3. When asked, upload your A2 `.nam` file(s). Then upload a training signal, or skip that step (see below).
4. The converted files download automatically when training finishes.

**Training signal**
- *Hotone Ampero users:* upload `nam_input_wav.wav` from your Mac. It is in
  `~/Library/Application Support/com.hotone.mp380/wavs/`.
  The Ampero Editor creates it at the first NAM import. If it's missing, import any A1 `.nam` once, or skip the upload.
- *Everyone else:* upload NAM's standard `input.wav`, or any varied guitar DI recording of at least 10 s.
  If you skip the upload (press *Cancel*), a synthetic signal is used instead. That still works, but the result is less accurate on real playing.

Files you upload are processed on Google's servers and deleted when the session ends. Details: [github.com/jiivari/nam-a2-to-a1](https://github.com/jiivari/nam-a2-to-a1)

In [ ]:
#@title 1. Settings
#@markdown Training steps: 6000 is enough for most captures. Raise it (e.g. 20000) if the result says POOR.
steps = 6000 #@param {type:"integer"}

In [ ]:
#@title 2. Install (about 2-3 minutes)
import os, subprocess, sys
REPO = "/content/nam-a2-to-a1"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/jiivari/nam-a2-to-a1.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
# Stage 1 (reads A2): neural-amp-modeler 0.13 in Colab's own Python
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-a2.txt"], check=True)
# Stage 2 (writes A1): neural-amp-modeler 0.11 in a separate environment that reuses Colab's PyTorch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
if not os.path.exists("/content/venv-a1/bin/python"):
    subprocess.run([sys.executable, "-m", "virtualenv", "-q", "--system-site-packages", "/content/venv-a1"], check=True)
subprocess.run(["/content/venv-a1/bin/pip", "install", "-q", "-r", f"{REPO}/requirements-a1.txt"], check=True)
import torch
print("Ready. GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")

In [ ]:
#@title 3. Upload A2 .nam file(s)
from google.colab import files
os.makedirs("/content/in", exist_ok=True)
up = files.upload()
models = []
for name, data in up.items():
    if name.lower().endswith(".nam"):
        p = f"/content/in/{name}"
        open(p, "wb").write(data)
        models.append(p)
print("Models:", [os.path.basename(m) for m in models] or "none - run this cell again")

In [ ]:
#@title 4. Upload training signal (.wav) - or press Cancel to use a synthetic signal
signal = None
try:
    up = files.upload()
    for name, data in up.items():
        if name.lower().endswith(".wav"):
            signal = f"/content/in/{name}"
            open(signal, "wb").write(data)
except Exception:
    pass
print("Training signal:", os.path.basename(signal) if signal else "synthetic")

In [ ]:
#@title 5. Convert
os.makedirs("/content/out", exist_ok=True)
os.makedirs("/content/work", exist_ok=True)
summary = []
for m in models:
    base = os.path.basename(m)[:-4]
    npz = f"/content/work/{base}.npz"
    out = f"/content/out/{base} (A1).nam"
    print(f"== {base}")
    cmd = [sys.executable, f"{REPO}/render_teacher.py", m, npz]
    if signal:
        cmd += ["--input", signal]
    if subprocess.run(cmd).returncode != 0:
        summary.append(f"FAILED  {base} (could not read the A2 file)")
        continue
    rc = subprocess.run(["/content/venv-a1/bin/python", f"{REPO}/a2_to_a1.py", npz, out,
                         "--steps", str(steps)]).returncode
    if rc == 0:
        summary.append(f"OK      {os.path.basename(out)}")
    elif rc == 2:
        bad = out.replace(" (A1).nam", " (A1) POOR - do not use.nam")
        os.replace(out, bad)
        summary.append(f"POOR    {os.path.basename(bad)}  (try more steps)")
    else:
        summary.append(f"FAILED  {base} (training error {rc})")
print("\nSummary:")
print("\n".join("  " + s for s in summary))

In [ ]:
#@title 6. Download results
for f in sorted(os.listdir("/content/out")):
    if f.endswith(".nam"):
        files.download(f"/content/out/{f}")